In [17]:
import pandas as pd
import numpy as np
from typing import Dict

In [2]:
df=pd.read_parquet("/home/rs/ml-projects/WDM_dataset/Features/chest/all_chest_heart.parquet")

In [3]:
X = df.drop(columns=["start_time", "end_time","subject","window_idx"])
y = df["label"]

In [4]:
print(X.columns)

Index(['label', 'chest_ecg_hr_mean', 'chest_ecg_hr_std', 'chest_ecg_rr_mean',
       'chest_ecg_rr_std', 'chest_ecg_rmssd', 'chest_ecg_nn50',
       'chest_ecg_pnn50', 'chest_ecg_lf_power', 'chest_ecg_hf_power',
       'chest_ecg_lf_hf_ratio'],
      dtype='object')


In [5]:
def get_corr_dict(df, target, method):
    return (
        df.corr(method=method)[target]
        .drop(target)
        .sort_values(ascending=False)
        .to_dict()
    )
    

In [6]:
corr_dict_pearson = get_corr_dict(X, "label", "pearson")
corr_dict_spearman = get_corr_dict(X, "label", "spearman")

In [7]:
print(corr_dict_pearson)
print(corr_dict_spearman)

{'chest_ecg_rr_std': 0.26061619654269236, 'chest_ecg_lf_power': 0.25143499198534963, 'chest_ecg_hr_std': 0.23196609537977397, 'chest_ecg_rmssd': 0.16457012374060048, 'chest_ecg_hf_power': 0.15214980442090148, 'chest_ecg_nn50': 0.11208479027542465, 'chest_ecg_pnn50': 0.09535529463419168, 'chest_ecg_lf_hf_ratio': 0.08438668533978685, 'chest_ecg_rr_mean': 0.06825549642442667, 'chest_ecg_hr_mean': -0.07936228547803263}
{'chest_ecg_lf_power': 0.27815980138517055, 'chest_ecg_hr_std': 0.274422192582574, 'chest_ecg_rr_std': 0.2598008759014329, 'chest_ecg_rmssd': 0.1493297723099622, 'chest_ecg_hf_power': 0.13838614946631073, 'chest_ecg_nn50': 0.10869122633806826, 'chest_ecg_lf_hf_ratio': 0.10501562982828465, 'chest_ecg_pnn50': 0.10216188490952018, 'chest_ecg_rr_mean': 0.023758091132619893, 'chest_ecg_hr_mean': -0.009055458559953673}


In [30]:
def print_dict(user_dict, title):
    print(title)
    print("-" * 35)

    for k, v in user_dict.items():
        print(f"{k:20} : {v:20}")

    print("-" * 35)

In [9]:
print("pearson correlation")
for k, v in corr_dict_pearson.items():
    print(f"{k:20} : {v:.4f}")

pearson correlation
chest_ecg_rr_std     : 0.2606
chest_ecg_lf_power   : 0.2514
chest_ecg_hr_std     : 0.2320
chest_ecg_rmssd      : 0.1646
chest_ecg_hf_power   : 0.1521
chest_ecg_nn50       : 0.1121
chest_ecg_pnn50      : 0.0954
chest_ecg_lf_hf_ratio : 0.0844
chest_ecg_rr_mean    : 0.0683
chest_ecg_hr_mean    : -0.0794


In [10]:
print("spearman correlation")
for k, v in corr_dict_spearman.items():
    print(f"{k:20} : {v:.4f}")

spearman correlation
chest_ecg_lf_power   : 0.2782
chest_ecg_hr_std     : 0.2744
chest_ecg_rr_std     : 0.2598
chest_ecg_rmssd      : 0.1493
chest_ecg_hf_power   : 0.1384
chest_ecg_nn50       : 0.1087
chest_ecg_lf_hf_ratio : 0.1050
chest_ecg_pnn50      : 0.1022
chest_ecg_rr_mean    : 0.0238
chest_ecg_hr_mean    : -0.0091


In [26]:
def categorize_features(pearson_correlation, spearman_correlation, threshold):
    feature_behavior = {}

    for f in pearson_correlation:
        p = pearson_correlation[f]
        s = spearman_correlation[f]

        # checking direction
        same_direction = (p * s > 0)
        strong_signal = (abs(p) > threshold) or (abs(s) > threshold)

        if same_direction and strong_signal:
            if p > 0:
                feature_behavior[f] = "increasing"
            else:
                feature_behavior[f] = "decreasing"
        else:
            feature_behavior[f] = "weak"

    return feature_behavior

In [27]:
def get_threshold(pearson_corr, spearman_corr, percentile):
    all_values = []

    for f in pearson_corr:
        all_values.append(abs(pearson_corr[f]))
        all_values.append(abs(spearman_corr[f]))

    return np.percentile(all_values, percentile)
threshold = get_threshold(corr_dict_pearson, corr_dict_spearman, 60)

In [28]:
feature_behaviour: Dict[str, str] = {}
feature_behaviour=categorize_features(corr_dict_pearson,corr_dict_spearman,threshold)

In [31]:
print_dict(feature_behaviour,"Behaviour")

Behaviour
-----------------------------------
chest_ecg_rr_std     : increasing          
chest_ecg_lf_power   : increasing          
chest_ecg_hr_std     : increasing          
chest_ecg_rmssd      : increasing          
chest_ecg_hf_power   : increasing          
chest_ecg_nn50       : weak                
chest_ecg_pnn50      : weak                
chest_ecg_lf_hf_ratio : weak                
chest_ecg_rr_mean    : weak                
chest_ecg_hr_mean    : weak                
-----------------------------------
